# NASA POWER (Izmir, TR) — solar/wind resource processing

Source: NASA POWER hourly API (`community=RE`), lat 38.4192 / lon 27.1287 (Izmir), 2018-01-01 to 2022-12-31. Public API, no auth, downloaded with:

```
https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=T2M,T2MDEW,RH2M,QV2M,TS,PS,WS10M,WS50M,WD10M,PRECTOTCORR,CLRSKY_SFC_SW_DWN,ALLSKY_SFC_LW_DWN,ALLSKY_SFC_PAR_TOT,ALLSKY_SFC_SW_DWN&community=RE&longitude=27.1287&latitude=38.4192&start=20180101&end=20221231&format=CSV
```

**Target:** `ALLSKY_SFC_SW_DWN` (all-sky global horizontal irradiance, Wh/m^2 — solar generation proxy).

**Deliberately excluded from the request:** `ALLSKY_SFC_SW_DNI` and `ALLSKY_SFC_SW_DIFF`. GHI is the near-deterministic radiative-transfer decomposition `DWN ~= DNI * cos(zenith) + DIFF` — including them as features would let the model reconstruct the target algebraically instead of forecasting it. This is the same class of problem as the OPSD submeter/grid-import relationship, just from physics instead of accounting.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
from common import report_candidate

RAW_PATH = "../data/raw/nasa_power_izmir.csv"
PROCESSED_PATH = "../data/processed/nasa_power_izmir.csv"

with open(RAW_PATH, encoding="utf-8") as f:
    lines = f.readlines()
header_end = next(i for i, l in enumerate(lines) if "END HEADER" in l)

df = pd.read_csv(RAW_PATH, skiprows=header_end + 1)
df["timestamp"] = pd.to_datetime(dict(year=df.YEAR, month=df.MO, day=df.DY, hour=df.HR))
df = df.set_index("timestamp").drop(columns=["YEAR", "MO", "DY", "HR"])
df.shape

In [ ]:
# NASA POWER uses -999 as a missing-data sentinel, not NaN -- must convert before any stats
sentinel_counts = (df == -999).sum()
print("sentinel (-999) count per column:")
print(sentinel_counts)
df = df.replace(-999, pd.NA).astype("float64")

In [ ]:
diffs = df.index.to_series().diff().dropna()
print(diffs.value_counts().head())

# small gaps (satellite processing lag) get interpolated, same bounded-limit discipline as the OPSD notebook
MAX_GAP_HOURS = 3
df = df.interpolate(method="time", limit=MAX_GAP_HOURS).dropna(how="any")
df.shape

In [ ]:
TARGET = "ALLSKY_SFC_SW_DWN"
feature_cols = [c for c in df.columns if c != TARGET]
report_candidate(df, TARGET, feature_cols, freq="1h", name="NASA POWER Izmir (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")